In [ ]:
r = 0
[r:=2*10*2**s+1+2*r for s in range(7)]

In [ ]:
from nn2 import *
def At(i, *_):
    return Res(RMSNorm(i), Attn(i, *_))
def Ff(i, *_):
    return Res(RMSNorm(i), SwiGLU(i, *_))
def U(i, *_):
    return Res(
        Linear(i, i, 0),
        T(MaxPool1d(2)),
        Linear(i, i, 0),
        *_,
        Linear(i, i*2),
        Unflatten(-1, (2,-1)),
        Flatten(-3, -2),
    )

In [ ]:
i,h,G,g,H = 64,16,4,1,256
mdl = Sequential(
    Zip(
        Embedding(256, i),
        Linear(i, i, 0),
        Linear(i, i, 0),
    ),
    Sequential(
        At(i, h, G, g), Ff(i, H),
        U(i,
            At(i, h, G, g), Ff(i, H),
            U(i,
                At(i, h, G, g), Ff(i, H),
                U(i,
                    At(i, h, G, g), Ff(i, H),
                    U(i,
                        At(i, h, G, g), Ff(i, H),
                        U(i,
                            At(i, h, G, g), Ff(i, H),
                            U(i,
                                At(i, h, G, g), Ff(i, H),
                                U(i,
                                    At(i, h, G, g), Ff(i, H),
                                ),
                                At(i, h, G, g), Ff(i, H),
                            ),
                            At(i, h, G, g), Ff(i, H),
                        ),
                        At(i, h, G, g), Ff(i, H),
                    ),
                    At(i, h, G, g), Ff(i, H),
                ),
                At(i, h, G, g), Ff(i, H),
            ),
            At(i, h, G, g), Ff(i, H),
        ),
        At(i, h, G, g), Ff(i, H),
    ),
    SwiGLU(i, H, 256),
).to('cuda',torch.bfloat16)
mdl[1] = torch.compile(mdl[1])
opt = torch.optim.Adam(mdl.parameters(), fused=True)
# lr = LRFinder(mdl, opt, lambda y,t:functional.cross_entropy(y.mT,t))
# lr.range_test(Dl(), start_lr=1e-5, end_lr=1e-3, num_iter=20)
lr = 1e-3 #lr.history['lr'][np.gradient(lr.history['loss']).argmin()]
lr = torch.optim.lr_scheduler.OneCycleLR(opt, lr, 100000, cycle_momentum=False)
plot = Plot()
for x,t in Dl():
    opt.zero_grad()
    y = []
    for i in range(2):
        x = mdl[:2]([y[-1].argmax(2), x, y[-1].softmax(2)@mdl[0][0].weight]if i else x)
        y += mdl[2](x),
    l = [functional.cross_entropy(y.mT,t)for y in y]
    reduce(torch.add,l).backward()
    # utils.clip_grad_norm_(mdl.parameters(), 1)
    # adaptive_clip_grad(mdl.parameters())
    opt.step()
    lr.step()
    plot(l=torch.stack(l).data, a=torch.stack([(y.argmax(2)==t).sum()/t.numel()for y in y]))